# Tutorial on deep learning
Using a neural network to assess vascular age from pulse waves

The material for this workshop has been adapted from supplementary material in this article https://doi.org/10.1093/ehjdh/ztab089.

This tutorial trains a deep learning neural network to classify PPG pulse waves as either "young" or "elderly". 
It uses publicly available, simulated PPG data (described in this article https://doi.org/10.1152/ajpheart.00218.2019).

The Pulse Wave Data Base (PWDB) has been developed by Peter Charlton and is described in this article https://journals.physiology.org/doi/full/10.1152/ajpheart.00218.2019.

The MATLAB code has been adapted in python for the needs of VITAL's first training school.

In [ ]:
from IPython.display import display, Image
from utilities.utils_tutorial2 import *
from scipy.io import loadmat

# PPG signals

Photoplethysmogram (PPG) pulse waves differ in shape between young and elderly subjects. Young subjects typically exhibit a second, diastolic peak on the downslope, which disappears with age and is not often present in PPGs from elderly subjects:

In [ ]:
display(Image(filename="ppg_signals.jpg", width=400, height=300))

We first decide whether to classify pulse waves as 'young' or 'elderly' or to predict the exact age of the subject.
This notebook supports **two modes**:
- **Classification:** Categorizing subjects as 'young' or 'elderly'.
- **Regression:** Predicting numerical age.

The mode is stored in a configuration dictionary.

In [ ]:
mode = get_config(mode='classification') # choose if regression or classification

# Load the PWDB data

In [ ]:
# Load the PWDB data
data = loadmat('data/pwdb_data.mat')
# Access the main data structure
pwdb_data = data['data']

# Step 1: Extract and downsample PPG data 

- Downsampling options:

In [ ]:
sampling_rate = 50  # new sampling frequency <- you choose

old_sampling_rate = pwdb_data['waves'][0, 0]['fs'][0, 0].flatten() # initial sampling frequency 500 Hz for PWDB
ds_factor = np.round(old_sampling_rate/sampling_rate) # downsampling factor

print(f"Downsampling from {old_sampling_rate[0]:.0f} Hz to {sampling_rate} Hz with a factor of {int(ds_factor[0])}.")

- Create the full dataset: load and downsample all PPG signals, along with age, subject_id, plausibility log of all virtual subjects.

In [ ]:
ppg_df = load_all_ppg_radial(pwdb_data, old_sampling_rate, sampling_rate)

In [ ]:
ppg_df.head() # this will allow you to preview the first lines of the pd.frame

# Step 2: Split the dataset
In the standard approach, datasets are split using randomized methods (e.g., sklearn's train_test_split). 
In this tutorial, we follow a simple structured splitting approach: 
    - even-indexed subjects go into the training+validation set, 
    - odd-indexed subjects go into the test set.

We have some further considerations:
- We need to discard implausible subjects, as per the plausibility_log column of the dataframe above.
- In the case of **classification** mode, we retain plausible subjects with the age of 25 (Young) and 75 (Elderly).
- In the case of **regression** mode, we retain the numerical age of all plausible subjects.


In [ ]:
train_data, test_data, test_data_for_postprocessing = split_train_test_data(ppg_df, mode="classification") # switch the mode [classification, regression] to obtain different subsets

# Inspect the pulse wave data

In [ ]:
train_data.head()

The predictor variable (the PPG) is contained in the column `ppg_signal`. The response variable is contained in the age column (in the case of classification, as categorical vectors of labels 25 and 75 years old). Here is an example of young and elderly PPG pulse waves from the simulated dataset:

In [ ]:
plot_pulse_waves(train_data, sampling_rate=50)

The pulse waves exhibited similar changes with age to those observed in healthy volunteers: most markedly, the diastolic peak disappears with age.

# Switch to tabular structure

### Expanding PPG Signals into Tabular Format

Since PPG signals are time-series data stored in a single column (`ppg_signal`), we expand them into a **structured tabular format**. Each time step of the signal is assigned to a **separate column**, allowing for easier preprocessing and integration with machine learning models.

This format

✔ Works well with standard machine learning models (e.g., XGBoost, Random Forest, Linear Models)  
✔ Avoids dealing with lists inside DataFrames  
✔ Enables direct feature engineering and scaling

In the next step, we
- Expand each time-series signal into separate numbered columns.
- Ensure the **age** column remains at the **end** for easy access (`X[:, -1]`).


In [ ]:
# Apply function to train and test datasets
train_data_expanded = switch_to_tabular(train_data, column_name="ppg_signal", target_column="age")
test_data_expanded = switch_to_tabular(test_data, column_name="ppg_signal", target_column="age")

print(f"Expanded DataFrame shape: {train_data_expanded.shape} (samples, time_steps + 1)")
train_data_expanded.head()

# Step 3: Scale the dataset

In [ ]:
X_train, X_test, y_train, y_test, scaler_X, scaler_y = scale_data(train_data_expanded, test_data_expanded)

In [ ]:
X_train.head()

- **Pad** shorter sequences with `-1` for uniform shape (this value is then filtered out in the LSTM).

In [ ]:
pad_value=-1
X_train=padding(X_train,pad_value)
X_train.head()

# Step 4: Defining a neural network
We now define the architecture of the neural network. A long short-term memory (LSTM) recurrent neural network is used, as this is suitable for time series data (in this case, PPG pulse waves).

In [ ]:
# Build the model dynamically with proper Masking
ppg_model = build_lstm_model(mode=mode)

# Print model summary
print(f"LSTM Model for {mode.capitalize()} Mode:")
ppg_model.summary()

In [ ]:
train_options = get_training_options(train_data, mode=mode)

print(f"Training Options for {mode.capitalize()} Mode:")
print(train_options)

# Step 5: Training the network


In [ ]:
trained_model, training_history = train_lstm_model(ppg_model, train_data, mode=mode)

# Step 6: Assessing performance on the testing data
Use the neural network to classify the subjects in the test dataset as either young or elderly.

In [ ]:
YPred = evaluate_model(trained_model, test_data, scaler_y, mode=mode)
print("Predictions:", YPred)

# Step 7 : Assess the performance of the neural network (only in classification mode)
Calculate the accuracy of classifications (the proportion of classifications which were correct).

In [ ]:
assess_model_performance(YPred, test_data, mode=mode)

Step 8: Investigate reasons for misclassifications

In [ ]:
analyze_classification_errors(YPred, test_data_for_postprocessing, pwdb_data)